In [2]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, StringType, StructType, StructField
import pytz
from datetime import datetime

# Set Spark configuration to handle ancient datetime values
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY")  # or "CORRECTED" based on your requirement
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")  # or "CORRECTED" based on your requirement

# Initialize log messages
log_messages = []

# Function to add log messages
def log(message):
    log_messages.append(message)
    print(message)

log("Checking if the staging table exists...")
# Check if the staging table exists, if not create it
try:
    DeltaTable.forName(spark, "DE_LH_BRONZE_ELC.G_L_Entry_IL")
    log("Staging Delta table exists.")
except:
    # Define schema for the staging table if it does not exist
    schema = main_df.schema  # Assuming schema of main table is available
    empty_df = spark.createDataFrame([], schema)
    empty_df.write.format("delta").saveAsTable("DE_LH_BRONZE_ELC.G_L_Entry_IL")
    log("Staging Delta table created.")

log("Checking if the log table exists...")
# Check if the log table exists, if not create it
try:
    DeltaTable.forName(spark, "DE_LH_BRONZE_ELC.G_L_Entry_Logs")
    log("Log Delta table exists.")
except:
    # Define schema for the log table
    log_schema = StructType([
        StructField("log_timestamp", TimestampType(), True),
        StructField("log_messages", StringType(), True)
    ])
    empty_log_df = spark.createDataFrame([], log_schema)
    empty_log_df.write.format("delta").saveAsTable("DE_LH_BRONZE_ELC.G_L_Entry_Logs")
    log("Log Delta table created.")

log("Loading the main Delta table...")
# Load the main table as a Delta table
main_delta_table = DeltaTable.forName(spark, "DE_LH_BRONZE_ELC.G_L_Entry")
main_df = main_delta_table.toDF()
log("Main Delta table loaded.")

log("Loading the staging Delta table...")
staging_delta_table = DeltaTable.forName(spark, "DE_LH_BRONZE_ELC.G_L_Entry_IL")
staging_df = staging_delta_table.toDF()
log("Staging Delta table loaded.")

# Initialize staging_df_cleaned
staging_df_cleaned = staging_df

# Align column names in staging_df_cleaned to match main_df
for col in staging_df_cleaned.columns:
    if col not in main_df.columns:
        new_col = col.replace('-', '_').replace(' ', '_')
        staging_df_cleaned = staging_df_cleaned.withColumnRenamed(col, new_col)

# Ensure all columns match between main_df and staging_df_cleaned
staging_df_cleaned = staging_df_cleaned.select(*main_df.columns)

# Count records before merge
main_count_before_merge = main_df.count()
log(f"Number of records in the main table before merge: {main_count_before_merge}")

# Identify records to be updated (primary key fields)
updated_records = staging_df_cleaned.join(main_df, ["Entry_No_", "Posting_Date"]).count()
log(f"Number of records to be updated: {updated_records}")

# Identify new records
new_records = staging_df_cleaned.join(main_df, ["Entry_No_", "Posting_Date"], "left_anti").count()
log(f"Number of new records to be inserted: {new_records}")

log("Starting the merge operation...")
# Merge the staging data into the main Delta table
main_delta_table.alias("main").merge(
    staging_df_cleaned.alias("staging"),
    "main.Entry_No_ = staging.Entry_No_ AND main.Posting_Date = staging.Posting_Date"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
log("Merge operation completed.")

# Count records after merge
main_count_after_merge = main_delta_table.toDF().count()
log(f"Number of records in the main table after merge: {main_count_after_merge}")

# Convert log messages to a single row DataFrame
log_timestamp = datetime.now(pytz.timezone('America/New_York'))
log_df = spark.createDataFrame(
    [(log_timestamp, "\n".join(log_messages))],
    schema=["log_timestamp", "log_messages"]
)

# Write log messages to the log Delta table
log_df.write.format("delta").mode("append").saveAsTable("DE_LH_BRONZE_ELC.G_L_Entry_Logs")

log("Log messages saved to the log Delta table.")

StatementMeta(, e12fd137-1bfe-4c67-b35e-2f48ba963ed4, 4, Finished, Available, Finished)

Checking if the staging table exists...
Staging Delta table exists.
Checking if the log table exists...
Log Delta table exists.
Loading the main Delta table...
Main Delta table loaded.
Loading the staging Delta table...
Staging Delta table loaded.
Number of records in the main table before merge: 107980055
Number of records to be updated: 5938
Number of new records to be inserted: 46273
Starting the merge operation...
Merge operation completed.
Number of records in the main table after merge: 108026328
Log messages saved to the log Delta table.
